# database.db 조회

`data_2026` 폴더의 `database.db`(DuckDB 파일)를 읽기 전용으로 열어 표 구성과 예시 조회를 확인하는 노트북입니다. 위에서 아래로 한 번에 실행할 수 있습니다.

연결: `data_2026/database.db`를 읽기 전용으로 엽니다(상대 경로, 노트북과 같은 폴더에서 실행).

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('data_2026/database.db', read_only=True)

표 목록을 확인합니다.

In [ ]:
tables = con.sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main' ORDER BY table_name").df()
tables

표마다 컬럼 구성(이름·자료형)을 확인합니다.

In [ ]:
for name in tables['table_name']:
    print(f'[{name}]')
    display(con.sql(f'DESCRIBE "{name}"').df()[['column_name', 'column_type']])

표마다 행 수를 확인합니다.

In [ ]:
counts = pd.DataFrame(
    [(name, con.sql(f'SELECT COUNT(*) FROM "{name}"').fetchone()[0]) for name in tables['table_name']],
    columns=['table_name', 'row_count'],
)
counts

표마다 앞 5행을 미리 봅니다.

In [ ]:
for name in tables['table_name']:
    print(f'[{name}]')
    display(con.sql(f'SELECT * FROM "{name}" LIMIT 5').df())

조회 예시 1: 2026년 표에서 서울특별시의 영업/정상 음식점 중 앞 10건(조건 조회). 아래 `query`를 바꿔 다시 실행하면 됩니다.

In [ ]:
query = """
SELECT 시설명, 구분, 시군구, 인허가일자
FROM restaurant_2026
WHERE 시도 = '서울특별시' AND 영업상태명 = '영업/정상'
ORDER BY 인허가일자 DESC
LIMIT 10
"""
con.sql(query).df()

조회 예시 2: 2026년 표에서 시도별 건수 상위 10곳(집계).

In [ ]:
query = """
SELECT 시도, COUNT(*) AS 건수
FROM restaurant_2026
GROUP BY 시도
ORDER BY 건수 DESC
LIMIT 10
"""
con.sql(query).df()

조회 예시 3: 연도별 표(2022~2026)의 영업상태별 건수(여러 표 합산 집계).

In [ ]:
query = """
SELECT 연도, 영업상태명, COUNT(*) AS 건수
FROM (
    SELECT 2022 AS 연도, 영업상태명 FROM restaurant_2022
    UNION ALL SELECT 2023, 영업상태명 FROM restaurant_2023
    UNION ALL SELECT 2024, 영업상태명 FROM restaurant_2024
    UNION ALL SELECT 2025, 영업상태명 FROM restaurant_2025
    UNION ALL SELECT 2026, 영업상태명 FROM restaurant_2026
)
GROUP BY 연도, 영업상태명
ORDER BY 연도, 건수 DESC
"""
con.sql(query).df()

연결을 닫습니다.

In [ ]:
con.close()